In [1]:
import os
from google.colab import drive

# Unmount if already mounted, and clear the mount point
if os.path.exists('/content/drive'):
    try:
        drive.flush_and_unmount()
    except ValueError:
        pass # Drive was not mounted, ignore error

    # Clear any files in the mount point if it exists
    for item in os.listdir('/content/drive'):
        item_path = os.path.join('/content/drive', item)
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.remove(item_path)
        elif os.path.isdir(item_path):
            import shutil
            shutil.rmtree(item_path)

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os
import random
import pandas as pd
from pathlib import Path
from PIL import Image
import shutil
import numpy as np
import cv2
import zipfile

# =========================
# OCR AUGMENTATION METHODS
# =========================

def thick_ocr(image):
    img_array = np.array(image)
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY) if img_array.ndim == 3 else img_array
    kernel = np.ones((2, 2), np.uint8)
    thickened = cv2.dilate(gray, kernel, iterations=1)
    if img_array.ndim == 3:
        thickened = cv2.cvtColor(thickened, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(thickened)

def thin_ocr(image):
    img_array = np.array(image)
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY) if img_array.ndim == 3 else img_array
    kernel = np.ones((2, 2), np.uint8)
    thinned = cv2.erode(gray, kernel, iterations=1)
    if img_array.ndim == 3:
        thinned = cv2.cvtColor(thinned, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(thinned)

# =========================
# LABEL LOADING HELPER
# =========================

def load_original_labels(train_dir):
    """
    Load labels from original Excel files in the train directory.
    Returns a dictionary mapping image_path to label for each folder.
    """
    train_path = Path(train_dir)
    label_maps = {}

    print("\n" + "=" * 60)
    print("LOADING ORIGINAL LABELS FROM EXCEL FILES")
    print("=" * 60)

    # Define the expected Excel files for each folder
    label_files = {
        'Letters': 'train_letter_labels.xlsx',
        'Words': 'train_word_labels.xlsx',
        'Paragraphs': 'train_paragraph_labels.xlsx'
    }

    for folder_name, excel_filename in label_files.items():
        folder_path = train_path / folder_name
        excel_path = folder_path / excel_filename

        if not excel_path.exists():
            print(f"\nWarning: {excel_path} not found. Trying .xls extension...")
            excel_path = folder_path / excel_filename.replace('.xlsx', '.xls')

        if excel_path.exists():
            print(f"\nReading: {excel_path}")
            try:
                df = pd.read_excel(excel_path)

                # Determine the correct image path column name
                image_path_col = None
                if 'Image Path' in df.columns:
                    image_path_col = 'Image Path'
                elif 'image_path' in df.columns:
                    image_path_col = 'image_path'

                if image_path_col is None:
                    print(f"  Warning: Neither 'Image Path' nor 'image_path' column found in {excel_filename}")
                    continue

                if 'Label' not in df.columns:
                    print(f"  Warning: 'Label' column not found in {excel_filename}")
                    continue

                # Create mapping from image_path to Label
                label_map = {}
                for _, row in df.iterrows():
                    img_path = str(row[image_path_col]) # Use the determined column name
                    label = row['Label']
                    label_map[img_path] = label

                label_maps[folder_name] = label_map
                print(f"  Loaded {len(label_map)} labels from {excel_filename}")

            except Exception as e:
                print(f"  Error reading {excel_filename}: {e}")
        else:
            print(f"\nWarning: Label file not found for {folder_name}")
            label_maps[folder_name] = {}

    print(f"\nTotal folders with labels loaded: {len(label_maps)}")
    return label_maps

def find_label_for_image(image_path, label_map):
    """
    Find the label for a given image path.
    Tries multiple matching strategies to handle path variations.
    """
    # Try exact match first
    if image_path in label_map:
        return label_map[image_path]

    # Try matching by filename only
    image_filename = Path(image_path).name
    for path, label in label_map.items():
        if Path(path).name == image_filename:
            return label

    # Try matching with normalized paths (forward slashes)
    normalized_path = image_path.replace('\\', '/')
    for path, label in label_map.items():
        if path.replace('\\', '/') == normalized_path:
            return label

    # Try partial match (ends with)
    for path, label in label_map.items():
        if path.endswith(image_filename) or image_path.endswith(Path(path).name):
            return label

    return None  # No label found

# =========================
# MAIN AUGMENTATION LOGIC
# =========================

def augment_training_data(
    train_dir,
    output_dir,
    augmentation_ratio=0.75,
    rotate_plus5_ratio=0.30,
    rotate_minus5_ratio=0.30,
    thick_ratio=0.20,
    thin_ratio=0.20,
    seed=42
):
    random.seed(seed)
    np.random.seed(seed)

    train_path = Path(train_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # Load all original labels
    label_maps = load_original_labels(train_dir)

    print("\n" + "=" * 60)
    print("AUGMENTING TRAINING DATASET")
    print("=" * 60)

    for root_folder in train_path.iterdir():
        if not root_folder.is_dir():
            continue

        print(f"\nProcessing folder: {root_folder.name}")

        output_folder = output_path / root_folder.name
        output_folder.mkdir(parents=True, exist_ok=True)

        # Get the label map for this folder
        label_map = label_maps.get(root_folder.name, {})
        if not label_map:
            print(f"  Warning: No labels found for {root_folder.name}")

        # -------------------------
        # Collect images
        # -------------------------
        all_images = []
        # Case 1: Images directly in the root_folder (Structure 2)
        for img in root_folder.iterdir():
            if img.is_file() and img.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".tiff"]:
                all_images.append({
                    "path": img,
                    "participant": "",
                    "relative_path": Path(img.name)
                })

        # Case 2: Images in participant subfolders (Structure 1 and 3)
        for participant_dir in root_folder.iterdir():
            if participant_dir.is_dir():
                for img in participant_dir.rglob("*"):
                    if img.is_file() and img.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".tiff"]:
                        all_images.append({
                            "path": img,
                            "participant": participant_dir.name,
                            "relative_path": img.relative_to(participant_dir)
                        })

        total_images = len(all_images)
        print(f"  Found {total_images} images")

        if total_images == 0:
            print(f"  No images found in {root_folder.name}, skipping augmentation for this folder.")
            df = pd.DataFrame(columns=["image_path", "Label", "is_augmented", "augmentation_type", "source_image"])
            csv_path = output_folder / f"train_{root_folder.name.lower()}_labels_full.csv"
            df.to_csv(csv_path, index=False)
            print(f"  Saved empty CSV: {csv_path}")
            continue

        n_aug = int(total_images * augmentation_ratio)
        images_to_augment = random.sample(all_images, n_aug)
        random.shuffle(images_to_augment)

        n_rp = int(n_aug * rotate_plus5_ratio)
        n_rm = int(n_aug * rotate_minus5_ratio)
        n_thick = int(n_aug * thick_ratio)

        rotate_plus = images_to_augment[:n_rp]
        rotate_minus = images_to_augment[n_rp:n_rp + n_rm]
        thick_imgs = images_to_augment[n_rp + n_rm:n_rp + n_rm + n_thick]
        thin_imgs = images_to_augment[n_rp + n_rm + n_thick:]

        # -------------------------
        # MASTER LABEL LIST
        # -------------------------
        label_rows = []
        labels_found = 0
        labels_missing = 0

        # -------------------------
        # Image Processing
        # -------------------------
        for img_info in all_images:
            img_path = img_info["path"]
            participant = img_info["participant"]
            rel_path = img_info["relative_path"]

            # Construct destination directory for the image
            if participant:
                dest_dir = output_folder / participant / rel_path.parent
            else:
                dest_dir = output_folder / rel_path.parent
            dest_dir.mkdir(parents=True, exist_ok=True)

            dest_original = dest_dir / img_path.name
            shutil.copy2(img_path, dest_original)

            # Construct original_path for CSV record
            if participant:
                original_path = str(Path(root_folder.name) / participant / rel_path)
            else:
                original_path = str(Path(root_folder.name) / rel_path)

            # Find label for this image
            label = find_label_for_image(original_path, label_map)
            if label is None:
                # Try alternate path formats
                alt_path = str(rel_path)
                label = find_label_for_image(alt_path, label_map)

            if label is not None:
                labels_found += 1
            else:
                labels_missing += 1
                print(f"    Warning: No label found for {original_path}")

            # Record ORIGINAL
            label_rows.append({
                "image_path": original_path,
                "Label": label,
                "is_augmented": False,
                "augmentation_type": "original",
                "source_image": original_path
            })

            if img_info in images_to_augment:
                img = Image.open(img_path)
                stem, ext = img_path.stem, img_path.suffix

                if img_info in rotate_plus:
                    aug_img = img.rotate(5, fillcolor="white")
                    aug_type = "rot_plus5"
                elif img_info in rotate_minus:
                    aug_img = img.rotate(-5, fillcolor="white")
                    aug_type = "rot_minus5"
                elif img_info in thick_imgs:
                    aug_img = thick_ocr(img)
                    aug_type = "thick"
                else:
                    aug_img = thin_ocr(img)
                    aug_type = "thin"

                aug_name = f"{stem}_aug_{aug_type}{ext}"
                dest_aug = dest_dir / aug_name
                aug_img.save(dest_aug)

                # Construct augmented_path for CSV record
                if participant:
                    augmented_path = str(Path(root_folder.name) / participant / rel_path.parent / aug_name)
                else:
                    if rel_path.parent != Path('.'):
                        augmented_path = str(Path(root_folder.name) / rel_path.parent / aug_name)
                    else:
                        augmented_path = str(Path(root_folder.name) / aug_name)

                # Record AUGMENTED (use same label as original)
                label_rows.append({
                    "image_path": augmented_path,
                    "Label": label,  # Same label as original
                    "is_augmented": True,
                    "augmentation_type": aug_type,
                    "source_image": original_path
                })

        # -------------------------
        # SAVE CSV
        # -------------------------
        df = pd.DataFrame(label_rows)
        csv_path = output_folder / f"train_{root_folder.name.lower()}_labels.csv"
        df.to_csv(csv_path, index=False)

        print(f"  Labels found: {labels_found}/{total_images}")
        print(f"  Labels missing: {labels_missing}/{total_images}")
        print(f"  Saved CSV: {csv_path}")
        print(f"  Total rows: {len(df)}")

    print("\n" + "=" * 60)
    print("Augmentation COMPLETE ✅")
    print("=" * 60)
    print(f"Output directory: {output_dir}")

# =========================
# RUN SCRIPT
# =========================

if __name__ == "__main__":
    train_directory = "/content/drive/MyDrive/Sample_split_dataset/train_dataset/train"
    augmented_output = "/content/drive/MyDrive/augmented_train"

    augment_training_data(
        train_dir=train_directory,
        output_dir=augmented_output,
        augmentation_ratio=0.75,
        rotate_plus5_ratio=0.30,
        rotate_minus5_ratio=0.30,
        thick_ratio=0.20,
        thin_ratio=0.20,
        seed=42
    )

    print("\n✓ Augmented dataset ready with labels")


LOADING ORIGINAL LABELS FROM EXCEL FILES

Reading: /content/drive/MyDrive/Sample_split_dataset/train_dataset/train/Letters/train_letter_labels.xlsx
  Loaded 951 labels from train_letter_labels.xlsx

Reading: /content/drive/MyDrive/Sample_split_dataset/train_dataset/train/Words/train_word_labels.xlsx
  Loaded 170 labels from train_word_labels.xlsx

Reading: /content/drive/MyDrive/Sample_split_dataset/train_dataset/train/Paragraphs/train_paragraph_labels.xlsx
  Loaded 17 labels from train_paragraph_labels.xlsx

Total folders with labels loaded: 3

AUGMENTING TRAINING DATASET

Processing folder: Paragraphs
  Found 17 images
  Labels found: 17/17
  Labels missing: 0/17
  Saved CSV: /content/drive/MyDrive/augmented_train/Paragraphs/train_paragraphs_labels.csv
  Total rows: 29

Processing folder: Letters
  Found 896 images
  Labels found: 896/896
  Labels missing: 0/896
  Saved CSV: /content/drive/MyDrive/augmented_train/Letters/train_letters_labels.csv
  Total rows: 1568

Processing folder

In [ ]:
import shutil

# Define the base name for the zip file (e.g., augmented_train)
zip_base_name = os.path.join(os.path.dirname(augmented_output), os.path.basename(augmented_output))

# Create a zip archive of the augmented_output directory
# The output will be augmented_train.zip in the same directory as augmented_output
shutil.make_archive(zip_base_name, 'zip', augmented_output)

print(f"\n✓ Augmented dataset zipped to: {zip_base_name}.zip")
print("You can now download this zip file from your Google Drive or Colab's file browser.")


✓ Augmented dataset zipped to: /content/drive/MyDrive/augmented_train.zip
You can now download this zip file from your Google Drive or Colab's file browser.
